# 17x_segmentation_design_260516

Segmentation design using 15x payment-removed OOF scores and 16x payment-removed SHAP evidence. No model training, Optuna, SHAP recalculation, feature removal, or campaign-final thresholding is performed.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib, os, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams
warnings.filterwarnings('ignore')

STEP = '17x_segmentation_design_260516'
PAYMENT_FEATURES = ['payment_is_mobile', 'payment_is_pc', 'payment_is_android', 'payment_is_ios']
PROXY_RULE_BLOCKLIST = set(PAYMENT_FEATURES + ['payment_device','age_group','is_female','is_male','gender','is_user_verified'])
EXPECTED_16X_COUNTS = {'overall_with_promotion': 76, 'overall_without_promotion': 75, 'promotion_only': 75, 'nonpromotion_only': 75}
RANDOM_STATE = 42

START = Path.cwd().resolve()
ROOT = None
PARK = None
for cand in [START] + list(START.parents):
    if cand.name == 'park.ingyeom' and (cand / 'note.md').exists():
        PARK = cand.resolve(); ROOT = cand.parent.resolve(); break
    if (cand / 'park.ingyeom' / 'note.md').exists():
        ROOT = cand.resolve(); PARK = (cand / 'park.ingyeom').resolve(); break
assert PARK is not None and PARK.name == 'park.ingyeom', f'Could not locate park.ingyeom from {START}'

NB_PATH = PARK / 'notebook' / STEP / f'{STEP}.ipynb'
OUT = PARK / 'reports' / 'segments' / STEP
FIG = PARK / 'reports' / 'figures' / STEP
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP}_review_package.zip'
NOTE = PARK / 'note.md'
for p in [NB_PATH.parent, OUT, FIG, ZIP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

exec_log = []
def log(msg):
    exec_log.append(f"{datetime.now().isoformat(timespec='seconds')} | {msg}")

log(f'START {STEP}')
log(f'cwd={START}')
log(f'park_root={PARK}')
for label, path in [('notebook_dir', NB_PATH.parent), ('output_dir', OUT), ('figure_dir', FIG), ('zip_path', ZIP_PATH)]:
    log(f'preexisting {label}: {path.exists()}')

def inside_park(path):
    return str(Path(path).resolve()).lower().startswith(str(PARK).lower())

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def stat_file(path):
    p = Path(path)
    return {'sha256': sha256_file(p), 'mtime': datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec='seconds'), 'size': p.stat().st_size}

def read_csv(path):
    return pd.read_csv(path)

def write_csv(df, name):
    path = OUT / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    log(f'created {path.name}: rows={len(df)}, cols={len(df.columns)}')
    return path

def final_checks_pass_or_warn(path):
    df = pd.read_csv(path)
    status_cols = [c for c in df.columns if c.lower() in {'status','result','check_status'}]
    if status_cols:
        vals = df[status_cols[0]].astype(str).str.lower()
        return not vals.str.fullmatch('fail|failed|error|critical_fail').any()
    joined = ' '.join(df.astype(str).fillna('').values.ravel()).lower()
    return 'fail' not in joined and 'error' not in joined

def q(series, prob):
    return float(pd.to_numeric(series, errors='coerce').quantile(prob))

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches='tight')
    plt.close()
    log(f'created figure {path.name}')

paths = {
    '06x_expanded_dataset': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515' / '06x_expanded_dataset.csv',
    '06x_conservative_dataset': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515' / '06x_conservative_dataset.csv',
    '06x_dataset_comparison_summary': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515' / '06x_dataset_comparison_summary.csv',
    '06x_final_checks': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515' / '06x_final_checks.csv',
    '06x_cold_start_hotfix_validation': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515' / '06x_cold_start_hotfix_validation.csv',
    '06x_model_feature_lists': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515' / '06x_model_feature_lists.csv',
    '07x_feature_mapping_master': PARK / 'reports' / 'audits' / '07x_feature_mapping_AARRR_260515' / '07x_feature_mapping_master.csv',
    '07x_expanded_AARRR_mapping': PARK / 'reports' / 'audits' / '07x_feature_mapping_AARRR_260515' / '07x_expanded_AARRR_mapping.csv',
    '07x_caveat_handoff': PARK / 'reports' / 'audits' / '07x_feature_mapping_AARRR_260515' / '07x_caveat_handoff.csv',
    '07x_scope_policy_handoff': PARK / 'reports' / 'audits' / '07x_feature_mapping_AARRR_260515' / '07x_scope_policy_handoff.csv',
    '07x_final_checks': PARK / 'reports' / 'audits' / '07x_feature_mapping_AARRR_260515' / '07x_final_checks.csv',
    '15x_oof_predictions': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_oof_predictions.csv',
    '15x_expanded_no_payment_device_feature_list': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_expanded_no_payment_device_feature_list.csv',
    '15x_model_summary_by_scope': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_model_summary_by_scope.csv',
    '15x_segment_risk_handoff': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_segment_risk_handoff.csv',
    '15x_open_risks_for_17x': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_open_risks_for_17x.csv',
    '15x_safe_unsafe_wording': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_safe_unsafe_wording.csv',
    '15x_proxy_artifact_audit': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_proxy_artifact_audit.csv',
    '15x_age40_unverified_ios_audit': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_age40_unverified_ios_audit.csv',
    '15x_final_checks': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_final_checks.csv',
    '16x_SHAP_candidate_plan': PARK / 'reports' / 'interpretation' / '16x_SHAP_candidate_interpretation_260516' / '16x_SHAP_candidate_plan.csv',
    '16x_payment_removed_input_gate': PARK / 'reports' / 'interpretation' / '16x_SHAP_candidate_interpretation_260516' / '16x_payment_removed_input_gate.csv',
    '16x_payment_removed_feature_audit': PARK / 'reports' / 'interpretation' / '16x_SHAP_candidate_interpretation_260516' / '16x_payment_removed_feature_audit.csv',
    '16x_SHAP_global_importance': PARK / 'reports' / 'interpretation' / '16x_SHAP_candidate_interpretation_260516' / '16x_SHAP_global_importance.csv',
    '16x_SHAP_family_importance': PARK / 'reports' / 'interpretation' / '16x_SHAP_candidate_interpretation_260516' / '16x_SHAP_family_importance.csv',
    '16x_SHAP_direction_summary': PARK / 'reports' / 'interpretation' / '16x_SHAP_candidate_interpretation_260516' / '16x_SHAP_direction_summary.csv',
    '16x_safe_unsafe_wording': PARK / 'reports' / 'interpretation' / '16x_SHAP_candidate_interpretation_260516' / '16x_safe_unsafe_wording.csv',
    '16x_open_risks_for_next_steps': PARK / 'reports' / 'interpretation' / '16x_SHAP_candidate_interpretation_260516' / '16x_open_risks_for_next_steps.csv',
    '16x_final_checks': PARK / 'reports' / 'interpretation' / '16x_SHAP_candidate_interpretation_260516' / '16x_final_checks.csv',
}
tenx_dir = PARK / 'reports' / 'audits' / '10x_feature_distribution_redundancy_pre_audit_260516'
tenx_files = sorted([p for p in tenx_dir.glob('*.csv') if any(k in p.name.lower() for k in ['final_checks','redundancy','distribution','caveat'])])
for p in tenx_files:
    paths['10x_' + p.stem] = p

pre_rows = []
loaded = {}
source_before = []
for key, path in paths.items():
    exists = path.exists()
    status = 'PASS' if exists else 'FAIL'
    detail = ''
    row_count = None; col_count = None; cols_present = True; loaded_ok = False
    if exists:
        st = stat_file(path)
        source_before.append({'file_path': str(path), 'file_role': key, 'sha256_before': st['sha256'], 'mtime_before': st['mtime'], 'size_before': st['size']})
        try:
            df = pd.read_csv(path)
            loaded[key] = df
            row_count = len(df); col_count = len(df.columns); loaded_ok = True
            if key.endswith('final_checks') and not final_checks_pass_or_warn(path):
                status = 'FAIL'; detail = 'upstream final_checks contains FAIL'
        except Exception as e:
            status = 'FAIL'; detail = repr(e)
    pre_rows.append({'input_group': key.split('_')[0], 'file_name': path.name, 'path': str(path), 'exists': exists, 'loaded': loaded_ok, 'row_count': row_count, 'column_count': col_count, 'required_columns_present': cols_present, 'status': status, 'detail': detail})
preflight = pd.DataFrame(pre_rows)

expanded = loaded.get('06x_expanded_dataset', pd.DataFrame()).copy()
feature_lists = loaded.get('06x_model_feature_lists', pd.DataFrame()).copy()
oof = loaded.get('15x_oof_predictions', pd.DataFrame()).copy()
gate16 = loaded.get('16x_payment_removed_input_gate', pd.DataFrame()).copy()
shap_global = loaded.get('16x_SHAP_global_importance', pd.DataFrame()).copy()

required_oof_cols = ['row_id','USER_KEY','feature_set_variant','dataset_scope','model_name','fold','is_repurchase','repurchase_score','churn_risk']
preflight.loc[preflight['file_name'].eq('15x_oof_predictions.csv'), 'required_columns_present'] = all(c in oof.columns for c in required_oof_cols)
expanded_feature_count = int(((feature_lists.get('feature_set_name') == 'expanded_feature_set') & (feature_lists.get('use_as_feature').astype(str).str.lower() == 'yes')).sum()) if len(feature_lists) else -1
oof_has_no_payment = len(oof) and (oof['feature_set_variant'].astype(str).eq('expanded_no_payment_device').any())
gate_count_ok = False
if len(gate16):
    gate_count_ok = all(int(gate16.loc[gate16['dataset_scope'].eq(scope), 'actual_SHAP_input_feature_count'].iloc[0]) == cnt for scope, cnt in EXPECTED_16X_COUNTS.items())
shap_payment_rows = int(shap_global['feature'].astype(str).str.contains('payment_is_|payment_device', case=False, na=False).sum()) if len(shap_global) and 'feature' in shap_global.columns else -1
write_csv(preflight, '17x_preflight_input_validation.csv')
log(f'expanded_shape={expanded.shape}; expanded_feature_count={expanded_feature_count}; 15x_no_payment={oof_has_no_payment}; 16x_gate_count_ok={gate_count_ok}; 16x_payment_rows={shap_payment_rows}')

primary = oof[(oof['feature_set_variant'].eq('expanded_no_payment_device')) & (oof['dataset_scope'].eq('overall_with_promotion')) & (oof['model_name'].eq('LightGBM'))].copy()
primary = primary.sort_values('row_id').reset_index(drop=True)
secondary = oof[(oof['feature_set_variant'].eq('expanded_no_payment_device')) & ~(oof['dataset_scope'].eq('overall_with_promotion') & oof['model_name'].eq('LightGBM'))].copy()
row_id_ok = len(primary) == len(expanded) == 23079 and primary['row_id'].tolist() == list(range(23079))
expanded = expanded.reset_index(drop=True).copy()
expanded.insert(0, 'row_id', np.arange(len(expanded)))
user_match = row_id_ok and primary['USER_KEY'].astype(str).tolist() == expanded['USER_KEY'].astype(str).tolist()
target_match = row_id_ok and primary['is_repurchase'].astype(int).tolist() == expanded['is_repurchase'].astype(int).tolist()
risk_ok = len(primary) and np.allclose(primary['churn_risk'].astype(float), 1 - primary['repurchase_score'].astype(float), atol=1e-10)

score_source = pd.DataFrame([{
    'selected_score_source_step': '15x_payment_device_sensitivity_260516',
    'source_file': str(paths['15x_oof_predictions']),
    'feature_set_variant': 'expanded_no_payment_device',
    'dataset_scope': 'overall_with_promotion',
    'model_name': 'LightGBM',
    'row_count': len(primary),
    'score_column': 'repurchase_score',
    'risk_column': 'churn_risk',
    'positive_class': 'is_repurchase=1',
    'payment_removed': 'yes',
    'selection_reason': 'Align 17x representative segmentation score with 16x payment-removed LightGBM SHAP evidence; not final model selection.',
    'fallback_used': 'no',
    'status': 'PASS' if row_id_ok and user_match and target_match and risk_ok else 'FAIL',
    'caution': 'OOF diagnostic score for segmentation design, not final campaign threshold.'
}])
for scope in ['overall_without_promotion','promotion_only','nonpromotion_only']:
    sub = oof[(oof['feature_set_variant'].eq('expanded_no_payment_device')) & (oof['dataset_scope'].eq(scope))]
    score_source = pd.concat([score_source, pd.DataFrame([{
        'selected_score_source_step': '15x_payment_device_sensitivity_260516', 'source_file': str(paths['15x_oof_predictions']), 'feature_set_variant': 'expanded_no_payment_device', 'dataset_scope': scope, 'model_name': 'diagnostic_multiple_models', 'row_count': len(sub), 'score_column': 'repurchase_score', 'risk_column': 'churn_risk', 'positive_class': 'is_repurchase=1', 'payment_removed': 'yes', 'selection_reason': 'Secondary diagnostic only; not mixed into representative row-level assignment.', 'fallback_used': 'no', 'status': 'PASS' if len(sub) > 0 else 'WARN', 'caution': 'Scope-specific diagnostic table only.'
    }])], ignore_index=True)
write_csv(score_source, '17x_score_source_selection.csv')

base = expanded.merge(primary[['row_id','repurchase_score','churn_risk']], on='row_id', how='left', validate='one_to_one')
base['risk_rank_desc'] = base['churn_risk'].rank(method='first', ascending=False).astype(int)
base['risk_percentile_desc'] = base['risk_rank_desc'] / len(base) * 100
genre_cols = [c for c in ['action_adventure_ratio','family_animation_ratio','drama_ratio','thriller_crime_ratio','sf_fantasy_ratio','comedy_ratio','romance_ratio','horror_ratio','documentary_ratio','historical_war_ratio','other_ratio'] if c in base.columns]
base['max_genre_ratio'] = base[genre_cols].max(axis=1)
base['dominant_genre_proxy'] = base[genre_cols].idxmax(axis=1).str.replace('_ratio','', regex=False)

thr = {
    'low_activity_watch_time_q25': q(base['total_watch_time_min'], 0.25),
    'low_activity_watch_count_q25': q(base['total_watch_count'], 0.25),
    'heavy_user_watch_time_q75': q(base['total_watch_time_min'], 0.75),
    'heavy_user_watch_count_q75': q(base['total_watch_count'], 0.75),
    'genre_focused_max_genre_ratio_q75': q(base['max_genre_ratio'], 0.75),
    'new_movie_oriented_q75': q(base['new_movie_in_365d_ratio'], 0.75),
    'old_movie_oriented_q75': q(base['old_movie_ratio_5y'], 0.75),
    'week3_inactive_watch_time_zero': 0.0,
    'retention_decay_w3_ratio_lt_0_5': 0.5,
    'proxy_medium_ratio_vs_overall': 1.5,
    'proxy_high_ratio_vs_overall': 2.0,
}

base['flag_high_risk_top10'] = (base['risk_percentile_desc'] <= 10).astype(int)
base['flag_high_risk_top20'] = (base['risk_percentile_desc'] <= 20).astype(int)
base['flag_low_risk_stable'] = ((base['risk_percentile_desc'] >= 80) & (base['retention_w3_ratio'] >= base['retention_w2_ratio'].fillna(0))).astype(int)
base['flag_cold_start_weak'] = ((base['is_cold_start_3d_fixed'].eq(1)) | (base['is_cold_start_7d_fixed'].eq(1))).astype(int)
base['flag_strong_early_activation'] = ((base['watch_time_min_w1'] >= q(base['watch_time_min_w1'], 0.75)) | (base['watch_session_w1'] >= q(base['watch_session_w1'], 0.75))).astype(int)
base['flag_week3_inactive'] = ((base['watch_time_min_w3'] <= 0) | (base['watch_session_w3'] <= 0)).astype(int)
base['flag_only_w1'] = base['is_only_w1'].fillna(0).astype(int)
base['flag_week3_drop'] = ((base['diff_between_w3_w2'] < 0) & (base['watch_time_min_w3'] < base['watch_time_min_w2'])).astype(int)
base['flag_retention_decay'] = ((base['retention_w3_ratio'] < base['retention_w2_ratio']) | (base['retention_w3_ratio'] < thr['retention_decay_w3_ratio_lt_0_5'])).astype(int)
base['flag_retention_stable'] = ((base['retention_w3_ratio'] >= base['retention_w2_ratio']) & (base['retention_w3_ratio'] >= thr['retention_decay_w3_ratio_lt_0_5'])).astype(int)
base['flag_heavy_user'] = ((base['total_watch_time_min'] >= thr['heavy_user_watch_time_q75']) | (base['total_watch_count'] >= thr['heavy_user_watch_count_q75'])).astype(int)
base['flag_low_activity'] = ((base['total_watch_time_min'] <= thr['low_activity_watch_time_q25']) | (base['total_watch_count'] <= thr['low_activity_watch_count_q25'])).astype(int)
base['flag_genre_focused'] = (base['max_genre_ratio'] >= thr['genre_focused_max_genre_ratio_q75']).astype(int)
base['flag_new_movie_oriented'] = (base['new_movie_in_365d_ratio'] >= thr['new_movie_oriented_q75']).astype(int)
base['flag_old_movie_oriented'] = (base['old_movie_ratio_5y'] >= thr['old_movie_oriented_q75']).astype(int)
base['flag_age40_unverified_ios'] = ((base['age_group'].eq(40)) & (base['is_user_verified'].eq(0)) & (base['payment_is_ios'].eq(1))).astype(int)

flag_defs = [
    ('flag_high_risk_top10','risk','churn_risk descending top 10 percent','churn_risk,risk_percentile_desc','top10 fixed percentile','no','no','no','score-derived diagnostic'),
    ('flag_high_risk_top20','risk','churn_risk descending top 20 percent','churn_risk,risk_percentile_desc','top20 fixed percentile','no','yes','no','score-derived provisional threshold'),
    ('flag_low_risk_stable','risk_behavior','bottom 20 percent risk and stable week3 retention','churn_risk,retention_w2_ratio,retention_w3_ratio','bottom20 fixed percentile plus behavior','no','yes','no','not campaign-safe threshold'),
    ('flag_cold_start_weak','behavior','cold start fixed flags indicate weak activation','is_cold_start_3d_fixed,is_cold_start_7d_fixed','06x fixed flags','no','yes','no','behavior-only representative candidate'),
    ('flag_strong_early_activation','behavior','week1 watch time or session above q75','watch_time_min_w1,watch_session_w1','q75','no','no','no','diagnostic'),
    ('flag_week3_inactive','behavior','week3 watch time or sessions equal zero','watch_time_min_w3,watch_session_w3','fixed zero','no','yes','no','behavior-only representative candidate'),
    ('flag_only_w1','behavior','only week1 activity in day0-20 window','is_only_w1','06x flag','no','yes','no','behavior-only representative candidate'),
    ('flag_week3_drop','behavior','week3 usage below week2 and diff negative','watch_time_min_w2,watch_time_min_w3,diff_between_w3_w2','observed decrease','no','yes','no','behavior-only representative candidate'),
    ('flag_retention_decay','behavior','week3 retention below week2 or below 0.5','retention_w2_ratio,retention_w3_ratio','fixed 0.5 plus relative drop','no','yes','no','behavior-only representative candidate'),
    ('flag_retention_stable','behavior','week3 retention stable vs week2 and at least 0.5','retention_w2_ratio,retention_w3_ratio','fixed 0.5 plus relative comparison','no','yes','no','behavior-only representative candidate'),
    ('flag_heavy_user','behavior','total watch time or count above q75','total_watch_time_min,total_watch_count','q75','no','no','no','diagnostic'),
    ('flag_low_activity','behavior','total watch time or count below q25','total_watch_time_min,total_watch_count','q25','no','yes','no','behavior-only representative candidate'),
    ('flag_genre_focused','content_proxy','max genre ratio above q75','genre ratio columns,max_genre_ratio','q75','no','yes','no','Movie_Master mapping proxy'),
    ('flag_new_movie_oriented','content_proxy','new movie in 365d ratio above q75','new_movie_in_365d_ratio','q75','no','yes','no','content proxy'),
    ('flag_old_movie_oriented','content_proxy','old movie ratio 5y above q75','old_movie_ratio_5y','q75','no','yes','no','content proxy'),
    ('flag_age40_unverified_ios','proxy_audit','age_group == 40 AND is_user_verified == 0 AND payment_is_ios == 1','age_group,is_user_verified,payment_is_ios','fixed audit definition','no','no','yes','audit only; never representative rule'),
]
flag_def_df = pd.DataFrame(flag_defs, columns=['flag_name','flag_type','definition_text','columns_used','threshold_source','model_feature_yes_no','representative_rule_candidate_yes_no','audit_only_yes_no','caveat'])
write_csv(flag_def_df, '17x_internal_multiflag_definitions.csv')
flag_cols = flag_def_df['flag_name'].tolist()
write_csv(base[['row_id','USER_KEY'] + flag_cols], '17x_internal_multiflag_assignment.csv')

rules = pd.DataFrame([
    (1,'high_risk_week3_inactive_or_drop','flag_high_risk_top20 == 1 AND (flag_week3_inactive == 1 OR flag_week3_drop == 1 OR flag_retention_decay == 1)','flag_high_risk_top20,flag_week3_inactive,flag_week3_drop,flag_retention_decay'),
    (2,'high_risk_only_w1_or_cold_start_weak','flag_high_risk_top20 == 1 AND (flag_only_w1 == 1 OR flag_cold_start_weak == 1)','flag_high_risk_top20,flag_only_w1,flag_cold_start_weak'),
    (3,'high_risk_low_activity','flag_high_risk_top20 == 1 AND flag_low_activity == 1','flag_high_risk_top20,flag_low_activity'),
    (4,'medium_risk_retention_decay','flag_high_risk_top20 == 0 AND churn_risk top 20-50 percent AND flag_retention_decay == 1','risk_percentile_desc,flag_retention_decay'),
    (5,'content_preference_target_candidate','flag_high_risk_top20 == 0 AND flag_low_activity == 0 AND content proxy flag == 1','flag_low_activity,flag_genre_focused,flag_new_movie_oriented,flag_old_movie_oriented'),
    (6,'stable_retained_user','flag_low_risk_stable == 1','flag_low_risk_stable'),
    (7,'general_observation','no prior rule matched','none'),
], columns=['segment_priority','representative_segment','matched_rule_text','rule_features'])
rules['provisional_label_yes_no'] = 'yes'
rules['final_segment_name_yes_no'] = 'no'
rules['small_n_review_required'] = 'computed_after_assignment'
rules['uses_payment_auth_demographic_proxy'] = rules['rule_features'].str.contains('|'.join(PROXY_RULE_BLOCKLIST), regex=True)
write_csv(rules, '17x_representative_segment_rules.csv')

conditions = [
    (base['flag_high_risk_top20'].eq(1) & (base['flag_week3_inactive'].eq(1) | base['flag_week3_drop'].eq(1) | base['flag_retention_decay'].eq(1))),
    (base['flag_high_risk_top20'].eq(1) & (base['flag_only_w1'].eq(1) | base['flag_cold_start_weak'].eq(1))),
    (base['flag_high_risk_top20'].eq(1) & base['flag_low_activity'].eq(1)),
    (base['flag_high_risk_top20'].eq(0) & base['risk_percentile_desc'].gt(20) & base['risk_percentile_desc'].le(50) & base['flag_retention_decay'].eq(1)),
    (base['flag_high_risk_top20'].eq(0) & base['flag_low_activity'].eq(0) & (base['flag_genre_focused'].eq(1) | base['flag_new_movie_oriented'].eq(1) | base['flag_old_movie_oriented'].eq(1))),
    (base['flag_low_risk_stable'].eq(1)),
]
segments = rules['representative_segment'].tolist()
priorities = rules['segment_priority'].tolist()
texts = rules['matched_rule_text'].tolist()
base['representative_segment'] = np.select(conditions, segments[:6], default='general_observation')
base['segment_priority'] = np.select(conditions, priorities[:6], default=7).astype(int)
text_map = dict(zip(rules['representative_segment'], rules['matched_rule_text']))
base['matched_rule_name'] = base['representative_segment']
base['matched_rule_text'] = base['representative_segment'].map(text_map)

threshold_rows = [
    ('high_risk_top10','fixed percentile','churn_risk descending top 10 percent',10.0,'risk_percentile_desc <= 10'),
    ('high_risk_top20','fixed percentile','churn_risk descending top 20 percent',20.0,'risk_percentile_desc <= 20'),
    ('low_risk_bottom20','fixed percentile','churn_risk descending bottom 20 percent',80.0,'risk_percentile_desc >= 80'),
    ('low_activity_watch_time_q25','quantile', 'total_watch_time_min q25', thr['low_activity_watch_time_q25'],'computed on 06x expanded rows'),
    ('low_activity_watch_count_q25','quantile', 'total_watch_count q25', thr['low_activity_watch_count_q25'],'computed on 06x expanded rows'),
    ('heavy_user_watch_time_q75','quantile', 'total_watch_time_min q75', thr['heavy_user_watch_time_q75'],'computed on 06x expanded rows'),
    ('heavy_user_watch_count_q75','quantile', 'total_watch_count q75', thr['heavy_user_watch_count_q75'],'computed on 06x expanded rows'),
    ('genre_focused_max_genre_ratio_q75','quantile', 'max genre ratio q75', thr['genre_focused_max_genre_ratio_q75'],'Movie_Master category mapping proxy'),
    ('new_movie_oriented_q75','quantile', 'new_movie_in_365d_ratio q75', thr['new_movie_oriented_q75'],'content proxy'),
    ('old_movie_oriented_q75','quantile', 'old_movie_ratio_5y q75', thr['old_movie_oriented_q75'],'content proxy'),
    ('proxy_contamination_medium','fixed ratio', 'overall 대비 1.5x', 1.5,'audit only'),
    ('proxy_contamination_high','fixed ratio', 'overall 대비 2.0x', 2.0,'audit only'),
]
threshold_audit = pd.DataFrame(threshold_rows, columns=['threshold_name','threshold_type','definition','threshold_value','source_or_caution'])
write_csv(threshold_audit, '17x_threshold_audit.csv')

assignment_cols = ['row_id','USER_KEY','representative_segment','segment_priority','matched_rule_name','matched_rule_text','is_repurchase','repurchase_score','churn_risk','is_promotion']
write_csv(base[assignment_cols], '17x_representative_segment_assignment.csv')

key_behavior = ['total_watch_count','total_watch_time_min','watch_days','active_ratio','recency','watch_time_min_w1','watch_time_min_w2','watch_time_min_w3','watch_session_w1','watch_session_w2','watch_session_w3','retention_w2_ratio','retention_w3_ratio','diff_between_w2_w1','diff_between_w3_w2','watch_ratio_under_1m','watch_ratio_under_5m','is_cold_start_3d_fixed','is_cold_start_7d_fixed','genre_diversity_count','old_movie_ratio_5y','new_movie_in_365d_ratio','max_genre_ratio']
datamart_cols = ['row_id','USER_KEY','is_repurchase','is_promotion','repurchase_score','churn_risk','risk_rank_desc','risk_percentile_desc','representative_segment','segment_priority'] + flag_cols + key_behavior + PAYMENT_FEATURES + ['is_user_verified','age_group','is_female','is_male','dominant_genre_proxy']
write_csv(base[datamart_cols], '17x_segmentation_base_datamart.csv')

summary = base.groupby(['representative_segment','segment_priority'], as_index=False).agg(
    row_count=('row_id','count'), repurchase_rate=('is_repurchase','mean'), mean_repurchase_score=('repurchase_score','mean'), mean_churn_risk=('churn_risk','mean'), median_churn_risk=('churn_risk','median'), top10_risk_share=('flag_high_risk_top10','mean'), top20_risk_share=('flag_high_risk_top20','mean'), promotion_share=('is_promotion','mean'))
summary['row_share'] = summary['row_count'] / len(base)
summary['nonrepurchase_rate'] = 1 - summary['repurchase_rate']
summary['nonpromotion_share'] = 1 - summary['promotion_share']
summary['small_n_review_required'] = summary['row_count'] < max(100, len(base) * 0.01)
summary['interpretation_caution'] = 'Provisional segment design only; row-level subscription-event rows, not unique customer count.'
summary = summary[['representative_segment','segment_priority','row_count','row_share','repurchase_rate','nonrepurchase_rate','mean_repurchase_score','mean_churn_risk','median_churn_risk','top10_risk_share','top20_risk_share','promotion_share','nonpromotion_share','small_n_review_required','interpretation_caution']].sort_values('segment_priority')
write_csv(summary, '17x_segment_summary.csv')

profile_rows = []
for seg, g in base.groupby('representative_segment'):
    for f in key_behavior:
        s = pd.to_numeric(g[f], errors='coerce')
        profile_rows.append({'representative_segment': seg, 'feature': f, 'mean': s.mean(), 'median': s.median(), 'q25': s.quantile(0.25), 'q75': s.quantile(0.75)})
profile = pd.DataFrame(profile_rows)
write_csv(profile, '17x_segment_feature_profile.csv')

rule_feature_map = {
    'high_risk_week3_inactive_or_drop': ['watch_time_min_w3','watch_session_w3','diff_between_w3_w2','retention_w3_ratio','retention_w2_ratio'],
    'high_risk_only_w1_or_cold_start_weak': ['is_only_w1','is_cold_start_3d_fixed','is_cold_start_7d_fixed'],
    'high_risk_low_activity': ['total_watch_time_min','total_watch_count'],
    'medium_risk_retention_decay': ['retention_w2_ratio','retention_w3_ratio'],
    'content_preference_target_candidate': ['max_genre_ratio','new_movie_in_365d_ratio','old_movie_ratio_5y'] + genre_cols,
    'stable_retained_user': ['retention_w2_ratio','retention_w3_ratio'],
    'general_observation': []
}
evidence_rows = []
for seg, feats in rule_feature_map.items():
    for feat in feats:
        sg = shap_global[shap_global['feature'].eq(feat)].copy() if len(shap_global) else pd.DataFrame()
        if len(sg):
            for _, r in sg.iterrows():
                evidence_rows.append({'representative_segment': seg, 'rule_feature': feat, 'dataset_scope': r.get('dataset_scope'), 'model_name': r.get('model_name'), 'shap_rank': r.get('rank_in_scope'), 'mean_abs_shap': r.get('mean_abs_shap'), 'mean_signed_shap': r.get('mean_signed_shap'), 'status': 'linked_to_16x_SHAP_global_importance', 'caution': 'SHAP is fitted model explanation, not causal evidence.'})
        else:
            evidence_rows.append({'representative_segment': seg, 'rule_feature': feat, 'dataset_scope': '', 'model_name': '', 'shap_rank': '', 'mean_abs_shap': '', 'mean_signed_shap': '', 'status': 'missing_from_SHAP_global_importance', 'caution': 'Missing does not invalidate rule; SHAP is not causal evidence.'})
write_csv(pd.DataFrame(evidence_rows), '17x_segment_SHAP_evidence_link.csv')

overall_proxy = {
    'payment_is_ios_share': base['payment_is_ios'].mean(), 'payment_is_android_share': base['payment_is_android'].mean(), 'payment_is_mobile_share': base['payment_is_mobile'].mean(), 'payment_is_pc_share': base['payment_is_pc'].mean(), 'is_user_verified_0_share': (base['is_user_verified'] == 0).mean(), 'age_group_40_share': (base['age_group'] == 40).mean(), 'gender_missing_or_N_proxy_share': ((base['is_female'] == 0) & (base['is_male'] == 0)).mean(), 'flag_age40_unverified_ios_share': base['flag_age40_unverified_ios'].mean()
}
proxy_rows = []
for seg, g in base.groupby('representative_segment'):
    metrics = {'payment_is_ios_share': g['payment_is_ios'].mean(), 'payment_is_android_share': g['payment_is_android'].mean(), 'payment_is_mobile_share': g['payment_is_mobile'].mean(), 'payment_is_pc_share': g['payment_is_pc'].mean(), 'is_user_verified_0_share': (g['is_user_verified'] == 0).mean(), 'age_group_40_share': (g['age_group'] == 40).mean(), 'gender_missing_or_N_proxy_share': ((g['is_female'] == 0) & (g['is_male'] == 0)).mean(), 'flag_age40_unverified_ios_share': g['flag_age40_unverified_ios'].mean()}
    ratios = {k + '_ratio_vs_overall': (v / overall_proxy[k] if overall_proxy[k] > 0 else np.nan) for k, v in metrics.items()}
    max_ratio = np.nanmax([x for x in ratios.values() if pd.notna(x)] or [0])
    level = 'high' if max_ratio >= 2 and len(g) >= 100 else ('medium' if max_ratio >= 1.5 else 'low')
    row = {'representative_segment': seg, 'row_count': len(g), 'overall_share': len(g) / len(base), 'max_ratio_vs_overall': max_ratio, 'proxy_contamination_level': level}
    row.update(metrics); row.update({k+'_overall': v for k, v in overall_proxy.items()}); row.update(ratios)
    proxy_rows.append(row)
proxy_audit = pd.DataFrame(proxy_rows)
write_csv(proxy_audit, '17x_proxy_artifact_audit.csv')

age_rows = []
age_rows.append({'group_type': 'overall', 'group_value': 'all', 'row_count': len(base), 'flag_share': base['flag_age40_unverified_ios'].mean(), 'representative_rule_used': 'no'})
for seg, g in base.groupby('representative_segment'):
    age_rows.append({'group_type': 'segment', 'group_value': seg, 'row_count': len(g), 'flag_share': g['flag_age40_unverified_ios'].mean(), 'representative_rule_used': 'no'})
for promo, g in base.groupby('is_promotion'):
    age_rows.append({'group_type': 'is_promotion', 'group_value': int(promo), 'row_count': len(g), 'flag_share': g['flag_age40_unverified_ios'].mean(), 'representative_rule_used': 'no'})
for flag, g in [('risk_top10', base[base['flag_high_risk_top10'].eq(1)]), ('risk_top20', base[base['flag_high_risk_top20'].eq(1)])]:
    age_rows.append({'group_type': flag, 'group_value': flag, 'row_count': len(g), 'flag_share': g['flag_age40_unverified_ios'].mean(), 'representative_rule_used': 'no'})
write_csv(pd.DataFrame(age_rows), '17x_age40_unverified_ios_audit.csv')

actions = pd.DataFrame([
    ('high_risk_week3_inactive_or_drop','candidate only','day14-20 observed usage drop or week3 inactivity re-entry message candidate','No payment/auth/demographic proxy action.'),
    ('high_risk_only_w1_or_cold_start_weak','candidate only','onboarding or low-friction first-watch recommendation candidate','No payment/auth/demographic proxy action.'),
    ('high_risk_low_activity','candidate only','low-barrier recommendation or reminder candidate','No payment/auth/demographic proxy action.'),
    ('medium_risk_retention_decay','candidate only','week2-week3 retention decay prevention message candidate','No causal claim.'),
    ('content_preference_target_candidate','candidate only','similar genre or content curation candidate','Genre/content is Movie_Master mapping proxy.'),
    ('stable_retained_user','candidate only','maintenance, upsell, satisfaction management candidate','Do not over-message.'),
    ('general_observation','candidate only','general monitoring or additional information needed','No final campaign action.'),
], columns=['representative_segment','action_status','business_action_candidate','caution'])
write_csv(actions, '17x_business_action_candidates.csv')

safe_name = {s:s for s in rules['representative_segment']}
dashboard = base[['row_id','USER_KEY','is_promotion','is_repurchase','repurchase_score','churn_risk','risk_percentile_desc','representative_segment'] + flag_cols + ['total_watch_count','total_watch_time_min','watch_days','active_ratio','recency','watch_time_min_w1','watch_time_min_w2','watch_time_min_w3','retention_w2_ratio','retention_w3_ratio','dominant_genre_proxy','max_genre_ratio']].copy()
dashboard['safe_display_segment_name'] = dashboard['representative_segment'].map(safe_name)
dashboard['unsafe_wording_warning'] = 'Do not describe as unique customers, causal effect, or payment/auth/demographic segment.'
dashboard['dashboard_use_caution'] = 'Design handoff datamart only; threshold and segment labels are provisional.'
write_csv(dashboard, '17x_dashboard_handoff_datamart.csv')

safe_unsafe = pd.DataFrame([
    ('safe','day0~20 관측창 내 사용 감소','observed behavior wording'),('safe','subscription-event rows','row-level count wording'),('safe','model explanation','SHAP limit'),('safe','churn_risk 기준 상위 위험 row','score direction'),('safe','Movie_Master category mapping 기준 장르 proxy','content caveat'),('safe','payment/auth/demographic proxy audit','proxy audit only'),
    ('unsafe','고객 수','unique customer not verified'),('unsafe','iOS 고객이 충성도가 높다','payment proxy misuse'),('unsafe','40대 미인증 iOS 세그먼트','proxy segment naming forbidden'),('unsafe','SHAP이 원인이다','causal misuse'),('unsafe','100원딜이 이탈을 유발했다','promotion causality forbidden'),('unsafe','payment_device는 시청기기다','wrong field meaning'),('unsafe','4주차까지 보고 판단했다','observation window violation')
], columns=['wording_type','wording','reason'])
write_csv(safe_unsafe, '17x_safe_unsafe_wording.csv')

open_risks = pd.DataFrame([
    ('representative segment threshold is provisional','Do not use as final campaign threshold.'),('segment name is not final before user approval','Keep labels provisional.'),('payment/auth/demographic proxy audit only','Never use as representative segment rule.'),('SHAP is not causal evidence','Use only as model explanation link.'),('genre/content is mapping proxy','Mention Movie_Master category mapping caveat.'),('OOF score is not final campaign criterion','Separate decision gate required.'),('row-level analysis, not unique customer analysis','Use subscription-event rows wording.')
], columns=['risk','handling'])
write_csv(open_risks, '17x_open_risks.csv')

fig_warnings = []
fonts = [f.name for f in font_manager.fontManager.ttflist]
font_name = 'Malgun Gothic' if 'Malgun Gothic' in fonts else 'DejaVu Sans'
rcParams['font.family'] = font_name; rcParams['axes.unicode_minus'] = False
if font_name != 'Malgun Gothic': fig_warnings.append({'figure':'font','warning':'Malgun Gothic unavailable; used English fallback labels'})
try:
    plt.figure(figsize=(9,4)); plt.bar(summary['representative_segment'], summary['row_count'], color='#378ADD'); plt.xticks(rotation=35, ha='right'); plt.ylabel('row count'); plt.title('Segment row count'); savefig(FIG / '17x_fig_segment_row_count.png')
    plt.figure(figsize=(9,4)); plt.bar(summary['representative_segment'], summary['mean_churn_risk'], color='#D4537E'); plt.xticks(rotation=35, ha='right'); plt.ylabel('mean churn risk'); plt.title('Segment churn risk'); savefig(FIG / '17x_fig_segment_churn_risk.png')
    plt.figure(figsize=(9,4)); plt.bar(summary['representative_segment'], summary['repurchase_rate'], color='#1D9E75'); plt.xticks(rotation=35, ha='right'); plt.ylabel('repurchase rate'); plt.title('Segment repurchase rate'); savefig(FIG / '17x_fig_segment_repurchase_rate.png')
    plt.figure(figsize=(9,4)); plt.bar(proxy_audit['representative_segment'], proxy_audit['max_ratio_vs_overall'], color='#D4537E'); plt.axhline(1.5, color='gray', linestyle='--'); plt.axhline(2.0, color='black', linestyle='--'); plt.xticks(rotation=35, ha='right'); plt.ylabel('max ratio vs overall'); plt.title('Proxy contamination audit'); savefig(FIG / '17x_fig_proxy_contamination.png')
except Exception as e:
    fig_warnings.append({'figure':'segment_figures','warning':repr(e)})
fig_inv = pd.DataFrame([{'figure_file': p.name, 'path': str(p), 'size_bytes': p.stat().st_size, 'status': 'PASS' if p.stat().st_size > 0 else 'FAIL'} for p in sorted(FIG.glob('*.png'))])
write_csv(fig_inv, '17x_visualization_inventory.csv')
write_csv(pd.DataFrame(fig_warnings if fig_warnings else [{'figure':'none','warning':'no visualization warnings'}]), '17x_visualization_warnings.csv')

source_after = []
for row in source_before:
    p = Path(row['file_path']); st = stat_file(p)
    source_after.append({**row, 'sha256_after': st['sha256'], 'mtime_after': st['mtime'], 'size_after': st['size'], 'status': 'PASS' if row['sha256_before'] == st['sha256'] and row['size_before'] == st['size'] else 'FAIL'})
fingerprint = pd.DataFrame(source_after)
write_csv(fingerprint, '17x_source_fingerprint_before_after.csv')

readme = f'''# {STEP}

## Purpose
17x is segmentation design for the 100-won-deal OTT churn analysis project. It is not modeling, Optuna, SHAP recalculation, feature removal, or final campaign threshold selection.

## Inputs and score source
Primary score source is `15x_oof_predictions.csv` filtered to `feature_set_variant == expanded_no_payment_device`, `dataset_scope == overall_with_promotion`, and `model_name == LightGBM`. This aligns the segmentation score with 16x payment-removed LightGBM SHAP evidence. It is not final model selection. `churn_risk = 1 - repurchase_score`, and top-k risk is sorted by churn_risk descending.

## Representative segment principle
Each subscription-event row receives exactly one representative provisional segment by priority rule. Payment, auth, and demographic proxy variables are not used in representative rules. `flag_age40_unverified_ios` is audit only.

## Key counts
- 06x expanded rows: {len(expanded)}
- 06x expanded feature count from feature list: {expanded_feature_count}
- representative assignment rows: {len(base)}
- segment count: {summary['representative_segment'].nunique()}

## Interpretation caveats
SHAP is model explanation, not causal evidence. is_promotion is not interpreted causally. Row counts are subscription-event rows, not unique customers. Genre/content fields are Movie_Master category mapping proxies. Payment/auth/demographic proxy variables are audit/caveat only.
'''
(OUT / 'README.md').write_text(readme, encoding='utf-8')
log('created README.md')

marker = '## 2026-05-18 | 17x_segmentation_design_260516 completion'
note_text = NOTE.read_text(encoding='utf-8')
final_fail_placeholder = 'pending final checks before zip packaging'
if marker not in note_text:
    append = f'''

{marker}

17x_segmentation_design_260516을 수행했다. 이번 17x는 segmentation design 단계이며 모델링, Optuna, SHAP 재계산, feature removal, campaign final threshold 결정 단계가 아니다.

score source는 15x `15x_oof_predictions.csv`에서 `feature_set_variant == expanded_no_payment_device`, `dataset_scope == overall_with_promotion`, `model_name == LightGBM` 조건으로 필터링한 OOF score를 primary로 사용했다. 이 선택은 16x payment-removed SHAP candidate plan이 LightGBM 기준으로 수행되었기 때문에 segmentation score와 SHAP evidence의 모델 기준을 맞추기 위한 것이다. 최종 모델 확정이라는 뜻은 아니다.

`churn_risk = 1 - repurchase_score` 관계를 검증했고, top-k risk는 `churn_risk` 내림차순 기준으로 사용했다. 16x payment-removed SHAP은 segment rule feature와 연결하는 evidence로만 사용했으며 SHAP은 인과가 아니라 fitted model explanation이다.

대표 segment rule에는 `payment_is_*`, payment_device, age_group, gender/is_female/is_male, is_user_verified를 사용하지 않았다. `flag_age40_unverified_ios`는 audit only로 생성했고 representative segment assignment에는 사용하지 않았다. representative segment name은 provisional label이며 사용자 승인 전 final segment가 아니다.

이번 산출물은 row-level/subscription-event-level 분석이다. row count를 고객 수 또는 unique customer 수로 표현하면 안 된다.

생성 산출물: 17x_preflight_input_validation.csv, 17x_score_source_selection.csv, 17x_segmentation_base_datamart.csv, 17x_threshold_audit.csv, 17x_internal_multiflag_definitions.csv, 17x_internal_multiflag_assignment.csv, 17x_representative_segment_rules.csv, 17x_representative_segment_assignment.csv, 17x_segment_summary.csv, 17x_segment_feature_profile.csv, 17x_segment_SHAP_evidence_link.csv, 17x_proxy_artifact_audit.csv, 17x_age40_unverified_ios_audit.csv, 17x_business_action_candidates.csv, 17x_dashboard_handoff_datamart.csv, 17x_safe_unsafe_wording.csv, 17x_open_risks.csv, 17x_source_fingerprint_before_after.csv, 17x_final_checks.csv, README.md, note_tail_copy.md, 17x_execution_log.txt, 17x_review_zip_inventory.csv, review zip.

미해결 리스크: threshold와 segment label은 provisional이고, payment/auth/demographic proxy는 audit만 가능하다. OOF score는 final campaign 확정 기준이 아니며, genre/content는 mapping proxy다. 다음 단계에서는 17x 산출물을 기준으로 발표 또는 dashboard handoff 문구를 안전 표현으로만 정리해야 한다.
'''
    NOTE.write_text(note_text.rstrip() + append + '\n', encoding='utf-8')
log('updated note.md')
(OUT / 'note_tail_copy.md').write_text('\n'.join(NOTE.read_text(encoding='utf-8').splitlines()[-220:]) + '\n', encoding='utf-8')

checks = []
def add_check(name, status, detail=''):
    checks.append({'check_name': name, 'status': status, 'detail': detail})
def passfail(cond): return 'PASS' if bool(cond) else 'FAIL'
all_outputs = list(OUT.glob('*')) + list(FIG.glob('*')) + [NB_PATH, ZIP_PATH]
add_check('all_outputs_inside_park_ingyeom', passfail(all(inside_park(p) for p in all_outputs)), '')
add_check('raw_source_csv_not_modified', passfail(fingerprint['status'].eq('PASS').all()), '')
add_check('source_fingerprint_created', passfail((OUT / '17x_source_fingerprint_before_after.csv').exists()), '')
add_check('source_fingerprint_unchanged', passfail(fingerprint['status'].eq('PASS').all()), '')
add_check('notebook_exists', passfail(NB_PATH.exists()), str(NB_PATH))
add_check('notebook_executed', 'PASS', 'execution reached final checks cell')
add_check('06x_inputs_loaded', passfail(preflight[preflight['input_group'].eq('06x')]['status'].eq('PASS').all()), '')
add_check('06x_final_checks_pass', passfail(final_checks_pass_or_warn(paths['06x_final_checks'])), '')
add_check('07x_inputs_loaded', passfail(preflight[preflight['input_group'].eq('07x')]['status'].eq('PASS').all()), '')
add_check('07x_final_checks_pass', passfail(final_checks_pass_or_warn(paths['07x_final_checks'])), '')
add_check('15x_inputs_loaded', passfail(preflight[preflight['input_group'].eq('15x')]['status'].eq('PASS').all()), '')
add_check('15x_final_checks_pass', passfail(final_checks_pass_or_warn(paths['15x_final_checks'])), '')
add_check('16x_inputs_loaded', passfail(preflight[preflight['input_group'].eq('16x')]['status'].eq('PASS').all()), '')
add_check('16x_final_checks_pass', passfail(final_checks_pass_or_warn(paths['16x_final_checks'])), '')
add_check('score_source_filtered_to_expanded_no_payment_device', passfail(score_source.iloc[0]['feature_set_variant'] == 'expanded_no_payment_device'), '')
add_check('primary_score_scope_overall_with_promotion', passfail(score_source.iloc[0]['dataset_scope'] == 'overall_with_promotion'), '')
add_check('primary_score_model_LightGBM', passfail(score_source.iloc[0]['model_name'] == 'LightGBM'), '')
add_check('churn_risk_equals_1_minus_repurchase_score', passfail(risk_ok), '')
add_check('row_count_23079', passfail(len(base) == 23079 and row_id_ok and user_match and target_match), f'base={len(base)}, row_id_ok={row_id_ok}, user_match={user_match}, target_match={target_match}')
add_check('one_representative_segment_per_row', passfail(len(base) == len(base['row_id'].unique()) and base['representative_segment'].notna().all()), '')
add_check('no_payment_feature_used_in_representative_rule', passfail(not rules['rule_features'].str.contains('payment', case=False, regex=False).any()), '')
add_check('no_auth_feature_used_in_representative_rule', passfail(not rules['rule_features'].str.contains('is_user_verified', case=False, regex=False).any()), '')
add_check('no_demographic_feature_used_in_representative_rule', passfail(not rules['rule_features'].str.contains('age_group|is_female|is_male|gender', case=False, regex=True).any()), '')
add_check('flag_age40_unverified_ios_audit_only', passfail(flag_def_df.loc[flag_def_df['flag_name'].eq('flag_age40_unverified_ios'), 'audit_only_yes_no'].iloc[0] == 'yes' and not rules['rule_features'].str.contains('flag_age40_unverified_ios', regex=False).any()), '')
add_check('payment_features_audit_only', passfail(not rules['rule_features'].str.contains('payment', case=False, regex=False).any()), '')
add_check('no_model_training_performed', 'PASS', '17x uses existing 15x OOF predictions only')
add_check('no_optuna_performed', 'PASS', 'no tuning executed')
add_check('no_SHAP_recalculation_performed', 'PASS', '16x SHAP CSV read only')
add_check('no_feature_removal_performed', 'PASS', '17x did not alter feature contracts')
add_check('segment_summary_created', passfail((OUT / '17x_segment_summary.csv').exists()), '')
add_check('proxy_audit_created', passfail((OUT / '17x_proxy_artifact_audit.csv').exists()), '')
add_check('business_action_candidates_created', passfail((OUT / '17x_business_action_candidates.csv').exists()), '')
add_check('dashboard_handoff_created', passfail((OUT / '17x_dashboard_handoff_datamart.csv').exists()), '')
add_check('safe_unsafe_wording_created', passfail((OUT / '17x_safe_unsafe_wording.csv').exists()), '')
add_check('README_created', passfail((OUT / 'README.md').exists()), '')
add_check('note_md_updated', passfail(marker in NOTE.read_text(encoding='utf-8')), '')
add_check('review_zip_created', 'WARN', 'zip is created after final_checks and then inventory is refreshed')
add_check('review_zip_inventory_created', 'WARN', 'inventory is created after final_checks')
final_checks = pd.DataFrame(checks)
write_csv(final_checks, '17x_final_checks.csv')
fail_count = int(final_checks['status'].eq('FAIL').sum())
log(f'final_checks fail_count={fail_count}; warn_count={int(final_checks.status.eq("WARN").sum())}')

log('END notebook execution before packaging')
(OUT / '17x_execution_log.txt').write_text('\n'.join(exec_log) + '\n', encoding='utf-8')

review_items = []
for p in [NB_PATH] + sorted(OUT.glob('*')) + sorted(FIG.glob('*.png')):
    if p.exists() and p.is_file():
        review_items.append({'path': str(p), 'arcname': str(p.relative_to(PARK)), 'size_bytes': p.stat().st_size})
review_inv = pd.DataFrame(review_items)
review_inv.to_csv(OUT / '17x_review_zip_inventory.csv', index=False, encoding='utf-8-sig')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for item in review_items:
        z.write(item['path'], item['arcname'])
    z.write(OUT / '17x_review_zip_inventory.csv', str((OUT / '17x_review_zip_inventory.csv').relative_to(PARK)))
log(f'created review zip {ZIP_PATH} size={ZIP_PATH.stat().st_size}')

final_checks.loc[final_checks['check_name'].eq('review_zip_created'), ['status','detail']] = ['PASS', str(ZIP_PATH)]
final_checks.loc[final_checks['check_name'].eq('review_zip_inventory_created'), ['status','detail']] = ['PASS', str(OUT / '17x_review_zip_inventory.csv')]
write_csv(final_checks, '17x_final_checks.csv')
(OUT / '17x_execution_log.txt').write_text('\n'.join(exec_log) + '\n', encoding='utf-8')
print('17x complete')
print(final_checks['status'].value_counts().to_dict())
print(summary[['representative_segment','row_count']].to_string(index=False))
print(ZIP_PATH)


17x complete
{'PASS': 38}
              representative_segment  row_count
    high_risk_week3_inactive_or_drop       3793
high_risk_only_w1_or_cold_start_weak        265
              high_risk_low_activity        511
         medium_risk_retention_decay       3195
 content_preference_target_candidate       6195
                stable_retained_user       1224
                 general_observation       7896
C:\Code\ott-churn-prediction\park.ingyeom\zip\17x_segmentation_design_260516_review_package.zip
